In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


df = pd.read_csv("../data/train.csv")
df = df.drop(index=df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)].index)

# инженерия
df["TotalSF"]   = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["TotalBath"] = (df["FullBath"] + 0.5*df["HalfBath"]
                   + df["BsmtFullBath"] + 0.5*df["BsmtHalfBath"])
df["HouseAge"]  = df["YrSold"] - df["YearBuilt"]
df["Qual_x_SF"] = df["OverallQual"] * df["TotalSF"]

X = df.drop(columns=["Id","SalePrice"])
y = np.log1p(df["SalePrice"])

num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = X.select_dtypes("object").columns.tolist()

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Pipeline([
    ("prep", preprocess),
    ("reg",  Ridge(alpha=1.0)),
])
model.fit(X_train, y_train)

def rmse_log(y_true, y_pred_log):
    return np.sqrt(mean_squared_error(y_true, y_pred_log))

y_pred = model.predict(X_test)
print("R²:  ", round(r2_score(y_test, y_pred), 4))
print("RMSE:", round(rmse_log(y_test, y_pred), 4))

R²:   0.9064
RMSE: 0.1256


C:\Users\АуфПК\AppData\Local\Temp\ipykernel_19320\1591531653.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes("object").columns.tolist()
c:\Users\АуфПК\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 29, 31] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [4]:
import sys, os
sys.path.append(os.path.abspath("../"))   
from utils import log_result

log_result(
    model_name="LinReg",
    y_true=y_test,     
    y_pred=y_pred,        
    params={"numeric_pipe": "median", "categorical_pipe": "most_frequent + OneHotEncoder"},
    notes=""
)

✓ LinReg: RMSLE=0.0099, R²=0.9064, RMSE=$0


{'model': 'LinReg',
 'timestamp': '2026-11-05 14:26',
 'rmsle': 0.0099,
 'rmse': 0.13,
 'mae': 0.09,
 'r2': 0.9064,
 'mape_pct': 0.73,
 'params': '{"numeric_pipe": "median", "categorical_pipe": "most_frequent + OneHotEncoder"}',
 'notes': ''}